In [9]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

In [8]:

load_dotenv()  # Load environment variables from .env file
apikey = os.getenv("GEMINI_API_KEY")  # Ensure you have

In [11]:
# Initialize the Gemini client (Make sure GEMINI_API_KEY is set in your environment variables)
client = genai.Client(api_key=apikey)

# List of example text strings to be converted into embeddings
texts = [
    "How do I write a Python function?",
    "How can I create a class in Java?",
    "Which smartphone has the best camera?",
    "What is the best Android phone?",
    "Turn off the lights.",
    "Switch the lights off.",
    "Apple released a new iPhone.",
    "Apple is my favorite fruit."
]

# By Default Dimension = 3072, but you can control dimensions using output_dimensionality (e.g., 768)
result = client.models.embed_content(
    model="gemini-embedding-001",  # Note: text-embedding-004 is the recommended updated model name
    contents=texts,
    config=types.EmbedContentConfig(
        task_type="SEMANTIC_SIMILARITY",
        output_dimensionality=768  # Adjust this value as needed (e.g., 256, 512, 768, 3072)
    )
)

# Loop through each text and its corresponding generated embedding object
for text, embedding_obj in zip(texts, result.embeddings):
    # Retrieve the list of floating-point values representing the vector
    embedding = embedding_obj.values
    
    print("Text:", text)
    print("Embedding length:", len(embedding))
    print("Embeddings (first 5 values):", embedding[:5], "...")  # Sliced to keep console readable
    print("-" * 50)

Text: How do I write a Python function?
Embedding length: 768
Embeddings (first 5 values): [-0.007334515, -0.006950146, 0.018683976, -0.04260659, -0.013810179] ...
--------------------------------------------------
Text: How can I create a class in Java?
Embedding length: 768
Embeddings (first 5 values): [-5.7711837e-05, -0.0018929766, 0.0235113, -0.031758267, -0.019546092] ...
--------------------------------------------------
Text: Which smartphone has the best camera?
Embedding length: 768
Embeddings (first 5 values): [-0.033830017, 0.0027793483, -0.00040252667, -0.054872185, -0.03099611] ...
--------------------------------------------------
Text: What is the best Android phone?
Embedding length: 768
Embeddings (first 5 values): [-0.023936875, -0.006036481, 0.0027132786, -0.04855059, -0.01584965] ...
--------------------------------------------------
Text: Turn off the lights.
Embedding length: 768
Embeddings (first 5 values): [-0.008670239, 0.00078756333, -0.016090266, -0.0665405,

In [12]:
!pip install chromadb

  Using cached opentelemetry_sdk-1.45.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
  Using cached pydantic-2.14.2-py3-none-any.whl.metadata (137 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached opentelemetry_exporter_otlp_common-0.66b2-py3-none-any.whl.metadata (1.9 kB)
  Using cached opentelemetry_exporter_otlp_proto_common-1.45.2-py3-none-any.whl.metadata (1.8 kB)
  Using cached opentelemetry_proto-1.45.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached opentelemetry_api-1.45.2-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_semantic_conventions-0.66b2-py3-none-any.whl.metadata (2.4 kB)
  Using cached pydantic_core-2.50.2-cp314-cp314-win_amd64.whl.metadata (6.7 kB)
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import chromadb
from chromadb.config import Settings

chromadb_client = chromadb.Client(Settings(
    persist_directory="./chromadb",  # Directory to store the database
)
)

collection = chromadb_client.get_or_create_collection(name = "santhosh-my_collection")


embeddings = []
ids = []

for i, embedding_obj in enumerate(result.embeddings):
    embedding = embedding_obj.values
    embeddings.append(embedding)
    ids.append(f"doc_{i}")


collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings
)
print("Embeddings added to the collection successfully.")

Embeddings added to the collection successfully.


In [17]:
query_text = "Which phone has the best camera?"
query_embedding = client.models.embed_content(
    model = "gemini-embedding-001",
    contents=[query_text],
    config=types.EmbedContentConfig(
        task_type="SEMANTIC_SIMILARITY",
        output_dimensionality=768
    )
).embeddings[0].values

results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3  # Number of similar documents to retrieve
    )     


print("Query Results:",results["documents"])

Query Results: [['Which smartphone has the best camera?', 'What is the best Android phone?', 'Apple released a new iPhone.']]
